In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
PYTORCH_ENABLE_MPS_FALLBACK=1 

# =========================================================
# 1. Data loading
# =========================================================
def load_file(filepath):
    return pd.read_csv(filepath, header=None, sep=r"\s+").values


def load_group(filenames, prefix):
    loaded = [load_file(prefix / name) for name in filenames]
    return np.dstack(loaded)  # (samples, timesteps, features)


def load_dataset_group(group, prefix):
    filepath = prefix / group / "Inertial Signals"

    filenames = [
        f"total_acc_x_{group}.txt", f"total_acc_y_{group}.txt", f"total_acc_z_{group}.txt",
        f"body_acc_x_{group}.txt",  f"body_acc_y_{group}.txt",  f"body_acc_z_{group}.txt",
        f"body_gyro_x_{group}.txt", f"body_gyro_y_{group}.txt", f"body_gyro_z_{group}.txt",
    ]

    X = load_group(filenames, filepath)
    y = load_file(prefix / group / f"y_{group}.txt").squeeze().astype(int) - 1
    return X, y


def load_dataset(prefix="UCI HAR Dataset"):
    prefix = Path(prefix)

    trainX, trainy = load_dataset_group("train", prefix)
    testX, testy = load_dataset_group("test", prefix)

    print("Loaded shapes:")
    print("trainX:", trainX.shape, "trainy:", trainy.shape)
    print("testX: ", testX.shape, "testy: ", testy.shape)

    return trainX, trainy, testX, testy


# =========================================================
# 2. Dataset class
# =========================================================
class HARDataset(Dataset):
    """
    Input expected:
        X: (samples, timesteps, features)

    Converted to PyTorch Conv1d format:
        (samples, features, timesteps)
    """
    def __init__(self, X, y):
        X = np.transpose(X, (0, 2, 1))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =========================================================
# 3. Multi-receptive-field CNN
# =========================================================
class ConvBranch(nn.Module):
    def __init__(self, in_channels, kernel_size, dropout=0.5):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=64, kernel_size=kernel_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(kernel_size=2),
        )

    def forward(self, x):
        return self.block(x)


class MultiReceptiveFieldCNN(nn.Module):
    def __init__(self, n_features, n_timesteps, n_outputs):
        super().__init__()

        # Same input, different temporal receptive fields
        self.branch_k3 = ConvBranch(in_channels=n_features, kernel_size=3)
        self.branch_k5 = ConvBranch(in_channels=n_features, kernel_size=5)
        self.branch_k11 = ConvBranch(in_channels=n_features, kernel_size=11)

        # Infer flattened output size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, n_features, n_timesteps)

            out1 = self.branch_k3(dummy).reshape(1, -1)
            out2 = self.branch_k5(dummy).reshape(1, -1)
            out3 = self.branch_k11(dummy).reshape(1, -1)

            merged_dim = out1.shape[1] + out2.shape[1] + out3.shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(merged_dim, 100),
            nn.ReLU(),
            nn.Linear(100, n_outputs),
        )

    def forward(self, x):
        b1 = self.branch_k3(x).reshape(x.size(0), -1)
        b2 = self.branch_k5(x).reshape(x.size(0), -1)
        b3 = self.branch_k11(x).reshape(x.size(0), -1)

        merged = torch.cat([b1, b2, b3], dim=1)
        out = self.classifier(merged)
        return out


# =========================================================
# 4. Training and evaluation
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_n += xb.size(0)

    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_n += xb.size(0)

    return total_loss / total_n, total_correct / total_n


# =========================================================
# 5. Main fit/evaluate function
# =========================================================
def evaluate_model(trainX, trainy, testX, testy, epochs=10, batch_size=32, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

    n_timesteps = trainX.shape[1]
    n_features = trainX.shape[2]
    n_outputs = len(np.unique(trainy))

    train_dataset = HARDataset(trainX, trainy)
    test_dataset = HARDataset(testX, testy)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = MultiReceptiveFieldCNN(
        n_features=n_features,
        n_timesteps=n_timesteps,
        n_outputs=n_outputs
    ).to(device)

    try:
        model = torch.compile(model)
        print("Using torch.compile()")
    except Exception:
        print("torch.compile() unavailable, continuing without it")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
        )

    return model, test_acc



In [3]:

# =========================================================
# 6. Run directly
# =========================================================

trainX, trainy, testX, testy = load_dataset("data/UCI_HAR_Dataset")
model, test_acc = evaluate_model(trainX, trainy, testX, testy, epochs=20, batch_size=32, lr=1e-3)
print(f"\nFinal test accuracy: {test_acc:.4f}")

Loaded shapes:
trainX: (7352, 128, 9) trainy: (7352,)
testX:  (2947, 128, 9) testy:  (2947,)
Using torch.compile()
Epoch 01/20 | Train Loss: 0.3381 | Train Acc: 0.8636 | Test Loss: 0.3756 | Test Acc: 0.8911
Epoch 02/20 | Train Loss: 0.1258 | Train Acc: 0.9464 | Test Loss: 0.3245 | Test Acc: 0.9030
Epoch 03/20 | Train Loss: 0.1086 | Train Acc: 0.9514 | Test Loss: 0.3404 | Test Acc: 0.9050
Epoch 04/20 | Train Loss: 0.1123 | Train Acc: 0.9531 | Test Loss: 0.3206 | Test Acc: 0.9084
Epoch 05/20 | Train Loss: 0.1000 | Train Acc: 0.9562 | Test Loss: 0.3597 | Test Acc: 0.9148
Epoch 06/20 | Train Loss: 0.0944 | Train Acc: 0.9576 | Test Loss: 0.3442 | Test Acc: 0.9145
Epoch 07/20 | Train Loss: 0.1059 | Train Acc: 0.9551 | Test Loss: 0.4033 | Test Acc: 0.8945
Epoch 08/20 | Train Loss: 0.1004 | Train Acc: 0.9597 | Test Loss: 0.2220 | Test Acc: 0.9365
Epoch 09/20 | Train Loss: 0.0861 | Train Acc: 0.9622 | Test Loss: 0.2452 | Test Acc: 0.9257
Epoch 10/20 | Train Loss: 0.0786 | Train Acc: 0.9640 | Te